In [17]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from itertools import combinations
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


In [11]:
# Datasets to be merged

# 1. Load Datasets
df_disney = pd.read_csv('Datasets/disney_movies.csv')
df_titles = pd.read_csv('Datasets/titles.csv')

# 2. Normalize title strings for higher match rate
df_disney['clean_title'] = df_disney['movie_title'].str.strip().str.lower()
df_titles['clean_title'] = df_titles['title'].str.strip().str.lower()

# Drop duplicate title records in titles.csv to avoid cartesian blowup (keep highest voted)
df_titles_dedup = df_titles.sort_values(by='imdb_votes', ascending=False).drop_duplicates('clean_title')

# 3. Inner Merge: Combines box office data with ratings, runtimes, and genre lists
merged_df = pd.merge(
    df_disney,
    df_titles_dedup[['clean_title', 'genres', 'runtime', 'imdb_score', 'imdb_votes', 'tmdb_popularity']],
    on='clean_title',
    how='inner'
)

# 4. Clean genres into valid Python lists
def clean_genre_list(g):
    if pd.isna(g):
        return []
    try:
        parsed = ast.literal_eval(g)
        return [item.lower().strip() for item in parsed] if isinstance(parsed, list) else [str(g).lower().strip()]
    except:
        return [str(g).lower().strip()]

merged_df['genre_basket'] = merged_df['genres'].apply(clean_genre_list)

# 5. Clean missing numerical values for regression
merged_df = merged_df.dropna(subset=['runtime', 'imdb_score', 'imdb_votes', 'inflation_adjusted_gross'])

# Save the finalized dataset
merged_df.to_csv('disney_combined_clean.csv', index=False)
print("Successfully created clean merged dataset with shape:", merged_df.shape)


Successfully created clean merged dataset with shape: (223, 13)


In [18]:
# 1. Load and merge datasets
df_disney = pd.read_csv('Datasets/disney_movies.csv')
df_titles = pd.read_csv('Datasets/titles.csv')

df_disney['clean_title'] = df_disney['movie_title'].str.strip().str.lower()
df_titles['clean_title'] = df_titles['title'].str.strip().str.lower()
df_titles_dedup = df_titles.sort_values(by='imdb_votes', ascending=False).drop_duplicates('clean_title')

merged_df = pd.merge(
    df_disney,
    df_titles_dedup[['clean_title', 'genres']],
    on='clean_title',
    how='inner'
)

def parse_genres(g):
    if pd.isna(g): return []
    try:
        p = ast.literal_eval(g)
        return sorted(list(set([str(x).lower().strip() for x in p]))) if isinstance(p, list) else [str(g).lower().strip()]
    except:
        return [str(g).lower().strip()]

transactions = merged_df['genres'].apply(parse_genres).tolist()
N = len(transactions)
minsupp = 0.15

In [19]:
# --- STEP 1: Growing Itemsets k=1 ---
all_items = sorted(list(set([item for t in transactions for item in t])))
k1_rows = []
L1 = []
for item in all_items:
    supp = sum(1 for t in transactions if item in t) / N
    status = 'Keep' if supp >= minsupp else 'Drop'
    if status == 'Keep': L1.append(item)
    k1_rows.append({'Itemset': f"{{{item}}}", 'Support': round(supp, 3), 'Status': status})

print("=== Table k=1 ===")
print(pd.DataFrame(k1_rows).to_string(index=False))

=== Table k=1 ===
        Itemset  Support Status
       {action}    0.270   Keep
    {animation}    0.354   Keep
       {comedy}    0.659   Keep
        {crime}    0.071   Drop
{documentation}    0.058   Drop
        {drama}    0.376   Keep
     {european}    0.075   Drop
       {family}    0.805   Keep
      {fantasy}    0.394   Keep
      {history}    0.035   Drop
       {horror}    0.027   Drop
        {music}    0.066   Drop
      {romance}    0.195   Keep
        {scifi}    0.177   Keep
        {sport}    0.097   Drop
     {thriller}    0.053   Drop
          {war}    0.013   Drop
      {western}    0.027   Drop


In [20]:
# --- STEP 2: Growing Itemsets k=2 ---
cands_k2 = list(combinations(L1, 2))
k2_rows = []
L2 = []
for c in cands_k2:
    supp = sum(1 for t in transactions if set(c).issubset(set(t))) / N
    status = 'Keep' if supp >= minsupp else 'Drop'
    if status == 'Keep': L2.append(set(c))
    k2_rows.append({'Itemset': f"{{{c[0]}, {c[1]}}}", 'Support': round(supp, 3), 'Status': status})

print("\n=== Table k=2 ===")
print(pd.DataFrame(k2_rows).to_string(index=False))


=== Table k=2 ===
             Itemset  Support Status
 {action, animation}    0.075   Drop
    {action, comedy}    0.133   Drop
     {action, drama}    0.071   Drop
    {action, family}    0.195   Keep
   {action, fantasy}    0.080   Drop
   {action, romance}    0.027   Drop
     {action, scifi}    0.102   Drop
 {animation, comedy}    0.279   Keep
  {animation, drama}    0.133   Drop
 {animation, family}    0.354   Keep
{animation, fantasy}    0.230   Keep
{animation, romance}    0.075   Drop
  {animation, scifi}    0.066   Drop
     {comedy, drama}    0.230   Keep
    {comedy, family}    0.593   Keep
   {comedy, fantasy}    0.288   Keep
   {comedy, romance}    0.155   Keep
     {comedy, scifi}    0.102   Drop
     {drama, family}    0.288   Keep
    {drama, fantasy}    0.111   Drop
    {drama, romance}    0.071   Drop
      {drama, scifi}    0.044   Drop
   {family, fantasy}    0.363   Keep
   {family, romance}    0.164   Keep
     {family, scifi}    0.115   Drop
  {fantasy, romance

In [21]:
# --- STEP 3: Growing Itemsets k=3 with Apriori Property ---
items_in_L2 = sorted(list(set([x for s in L2 for x in s])))
cands_k3 = [c for c in combinations(items_in_L2, 3) if all(set(sub) in L2 for sub in combinations(c, 2))]
k3_rows = []
for c in cands_k3:
    supp = sum(1 for t in transactions if set(c).issubset(set(t))) / N
    status = 'Keep' if supp >= minsupp else 'Drop'
    k3_rows.append({'Itemset': f"{{{c[0]}, {c[1]}, {c[2]}}}", 'Support': round(supp, 3), 'Status': status})

print("\n=== Table k=3 ===")
print(pd.DataFrame(k3_rows).to_string(index=False))


=== Table k=3 ===
                     Itemset  Support Status
 {animation, comedy, family}    0.279   Keep
{animation, comedy, fantasy}    0.190   Keep
{animation, family, fantasy}    0.230   Keep
     {comedy, drama, family}    0.204   Keep
   {comedy, family, fantasy}    0.274   Keep
   {comedy, family, romance}    0.128   Drop


In [22]:
# --- STEP 4: Rule Generation for {animation, comedy, fantasy} ---
target_set = {'animation', 'comedy', 'fantasy'}
supp_target = sum(1 for t in transactions if target_set.issubset(set(t))) / N

def calc_metrics(ante, cons):
    s_ante = 1.0 if not ante else sum(1 for t in transactions if ante.issubset(set(t))) / N
    s_cons = 1.0 if not cons else sum(1 for t in transactions if cons.issubset(set(t))) / N
    conf = supp_target / s_ante
    lift = supp_target / (s_ante * s_cons)
    leverage = supp_target - (s_ante * s_cons)
    return round(conf, 2), round(lift, 2), round(leverage, 2)

In [23]:
rule_rows = []
# 1. Empty ante
c, l, lev = calc_metrics(set(), target_set)
rule_rows.append({'Rule': 'Ø => {animation, comedy, fantasy}', 'Confidence': c, 'Lift': l, 'Leverage': lev})

# 2. Size 1 ante
for x in sorted(target_set):
    ante = {x}
    cons = target_set - ante
    c, l, lev = calc_metrics(ante, cons)
    rule_rows.append({'Rule': f"{{{x}}} => {{{', '.join(sorted(cons))}}}", 'Confidence': c, 'Lift': l, 'Leverage': lev})

# 3. Size 2 ante
for pair in combinations(sorted(target_set), 2):
    ante = set(pair)
    cons = target_set - ante
    c, l, lev = calc_metrics(ante, cons)
    rule_rows.append({'Rule': f"{{{', '.join(sorted(ante))}}} => {{{list(cons)[0]}}}", 'Confidence': c, 'Lift': l, 'Leverage': lev})

# 4. Empty cons
c, l, lev = calc_metrics(target_set, set())
rule_rows.append({'Rule': '{animation, comedy, fantasy} => Ø', 'Confidence': c, 'Lift': l, 'Leverage': lev})

print("\n=== Overview of Scores (Lecture Slide 16 Match) ===")
print(pd.DataFrame(rule_rows).to_string(index=False))


=== Overview of Scores (Lecture Slide 16 Match) ===
                             Rule  Confidence  Lift  Leverage
Ø => {animation, comedy, fantasy}        0.19  1.00      0.00
 {animation} => {comedy, fantasy}        0.54  1.87      0.09
 {comedy} => {animation, fantasy}        0.29  1.25      0.04
 {fantasy} => {animation, comedy}        0.48  1.73      0.08
 {animation, comedy} => {fantasy}        0.68  1.73      0.08
 {animation, fantasy} => {comedy}        0.83  1.25      0.04
 {comedy, fantasy} => {animation}        0.66  1.87      0.09
{animation, comedy, fantasy} => Ø        1.00  1.00      0.00
